# FIRMS research data quality
Companion to `docs/research-training.md`. Run from the notebook directory or worktree root. This notebook checks acquisition quality and split metadata only; it does not evaluate held-out predictions or access credentials. Outputs are regenerated locally.

In [ ]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
annual = {year: pd.read_parquet(root / f'data/cleaned/firms-snpp-{year}-v1/observations.parquet') for year in (2024, 2025)}
frame = pd.concat(annual.values(), ignore_index=True)
stamp = pd.to_datetime(frame.timestamp_utc, utc=True)
pd.DataFrame({'rows': {year: len(value) for year, value in annual.items()}})

In [ ]:
quality = {'rows': len(frame), 'duplicate_event_ids': int(frame.event_id.duplicated().sum()), 'missing_by_column': frame.isna().sum().to_dict(), 'first_observation': str(stamp.min()), 'last_observation': str(stamp.max())}
quality

In [ ]:
days = pd.date_range('2024-01-01', '2025-12-31', tz='UTC')
empty_dates = days.difference(stamp.dt.floor('D').unique())
print('Dates with no detections:', len(empty_dates))
print([str(d.date()) for d in empty_dates])
# Empty dates do not imply that no fires existed.
frame.groupby([stamp.dt.year.rename('year'), stamp.dt.month.rename('month')]).size().rename('observations').to_frame()

In [ ]:
ordered = stamp.sort_values().reset_index(drop=True)
validation_start = ordered.iloc[int(len(ordered)*0.8)]
test_start = ordered.iloc[int(len(ordered)*0.9)]
{'train': int((stamp < validation_start).sum()), 'validation': int(((stamp >= validation_start) & (stamp < test_start)).sum()), 'test': int((stamp >= test_start).sum()), 'validation_start': str(validation_start), 'test_start': str(test_start)}

## Interpretation
FIRMS `type` is a provider flag, not confirmed accident truth. Generated industrial-fire labels depend on distance/history features also supplied to CatBoost, so their agreement metrics are circular. The autoencoder uses no target or provider-type field. Its reconstruction error is novelty, not classification accuracy. The two-year holdout overlaps part of an earlier one-year validation interval; it is not a pristine final benchmark. Preserve the original 2026 test cohort for any future final protocol.

In [ ]:
report = json.loads((root / 'data/runs/anomaly-snpp-2024-2025-v1/run_report.json').read_text())
{key: value for key, value in report.items() if key != 'history'}